In [ ]:
# Cell 1: Project 2 - House Price Prediction Data Generation
np.random.seed(42)
n_houses = 600

area_sqft = np.random.uniform(800, 4500, n_houses)
bedrooms = np.random.randint(1, 6, n_houses)
bathrooms = np.random.randint(1, 4, n_houses)
parking_spaces = np.random.randint(0, 3, n_houses)
age_years = np.random.uniform(1, 30, n_houses)
location = np.random.choice(['Urban', 'Suburban', 'Rural'], size=n_houses, p=[0.5, 0.3, 0.2])

# Calculate price based on features with random variance
loc_multiplier = {'Urban': 1.5, 'Suburban': 1.2, 'Rural': 0.9}
loc_factor = np.array([loc_multiplier[l] for l in location])

price = (area_sqft * 120 + bedrooms * 15000 + bathrooms * 20000 + parking_spaces * 10000 - age_years * 1500) * loc_factor
price += np.random.normal(0, 25000, n_houses)

df_house = pd.DataFrame({
    'Area_SqFt': np.round(area_sqft, 2),
    'Bedrooms': bedrooms,
    'Bathrooms': bathrooms,
    'Parking_Spaces': parking_spaces,
    'Age_Years': np.round(age_years, 1),
    'Location': location,
    'Price_USD': np.round(price, 2)
})

df_house.to_csv("house_price_data.csv", index=False)
print("Project 2 Dataset Shape:", df_house.shape)
df_house.head()

In [ ]:
# Cell 1: Project 2 - Regression Models Comparison
X_house = df_house.drop(columns=['Price_USD'])
y_house = df_house['Price_USD']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Feature Transformation Pipelines
numeric_features = ['Area_SqFt', 'Bedrooms', 'Bathrooms', 'Parking_Spaces', 'Age_Years']
categorical_features = ['Location']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

# Models to Compare
regressors = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

house_results = {}

for name, model in regressors.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train_h, y_train_h)
    y_pred = pipeline.predict(X_test_h)
    
    mae = mean_absolute_error(y_test_h, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_h, y_pred))
    r2 = r2_score(y_test_h, y_pred)
    
    house_results[name] = {
        'MAE': mae, 'RMSE': rmse, 'R2 Score': r2, 'Pipeline': pipeline
    }

df_res_house = pd.DataFrame(house_results).T[['MAE', 'RMSE', 'R2 Score']]
print("--- Project 2 Performance Comparison ---")
print(df_res_house)

In [ ]:
# Cell 1: Project 2 - Regression Visualization & Model Saving
best_house_model_name = df_res_house['R2 Score'].idxmax()
best_house_pipeline = house_results[best_house_model_name]['Pipeline']

y_pred_best_h = best_house_pipeline.predict(X_test_h)

# Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test_h, y_pred_best_h, alpha=0.6, color='teal')
plt.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 'r--', lw=2)
plt.title(f'Actual vs Predicted House Prices ({best_house_model_name})')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.tight_layout()
plt.show()

# Save Best Pipeline
joblib.dump(best_house_pipeline, "best_house_price_pipeline.pkl")
print(f"Saved complete pipeline as 'best_house_price_pipeline.pkl'")